# Sentiment Analysis on Twitter Dataset

## Mục tiêu
Phân loại cảm xúc tweet thành positive, negative, neutral.

Notebook gồm: EDA, Machine Learning (Logistic Regression, Naive Bayes, SVM), LLM (RoBERTa) và so sánh kết quả.

## Dataset Description
Dataset chứa tweet và nhãn sentiment.

- Train: 27,481 mẫu
- Test: 3,534 mẫu
- Nhãn: positive, negative, neutral

Ngoài nội dung tweet còn có thông tin thời gian, độ tuổi, quốc gia.

## Import thư viện
Pandas xử lý dữ liệu, Seaborn/Matplotlib trực quan hóa, Scikit-learn xây dựng mô hình.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Đọc dữ liệu
Train dùng để huấn luyện, Test dùng để đánh giá khả năng dự đoán trên dữ liệu chưa thấy.

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
print(train_df.shape)
print(test_df.shape)
train_df.head()

# Part A - Exploratory Data Analysis (EDA)
EDA giúp hiểu dữ liệu trước khi xây dựng mô hình.

## Thông tin dữ liệu
Kiểm tra số lượng cột, kiểu dữ liệu và thống kê mô tả.

In [ ]:
train_df.info()
train_df.describe(include="all")

## Kiểm tra Missing Values
Tìm giá trị bị thiếu để xử lý trước khi huấn luyện.

In [ ]:
train_df.isnull().sum()

## Phân bố nhãn cảm xúc
Kiểm tra dữ liệu có cân bằng giữa positive, neutral và negative hay không.

In [ ]:
train_df["sentiment"].value_counts()

In [ ]:
sns.countplot(data=train_df,x="sentiment")
plt.title("Sentiment Distribution")
plt.show()

## Độ dài Tweet
Phân tích số ký tự trong mỗi tweet để hiểu đặc điểm dữ liệu văn bản.

In [ ]:
train_df["length"] = train_df["text"].astype(str).apply(len)
sns.histplot(train_df["length"], bins=50)
plt.title("Tweet Length Distribution")
plt.show()

## Word Cloud
Từ xuất hiện nhiều sẽ hiển thị lớn hơn, giúp khám phá từ khóa phổ biến trong nhóm Positive.

In [ ]:
from wordcloud import WordCloud
positive_text=" ".join(train_df[train_df.sentiment=="positive"]["text"].astype(str))
wc=WordCloud(width=800,height=400,background_color="white").generate(positive_text)
plt.figure(figsize=(10,5))
plt.imshow(wc)
plt.axis("off")
plt.show()

# Part B - Machine Learning
Máy tính không hiểu văn bản trực tiếp nên cần chuyển text thành vector số bằng TF-IDF.

## TF-IDF Feature Extraction
TF-IDF đánh giá tầm quan trọng của từ trong tập dữ liệu.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=10000,stop_words="english")
X_train=vectorizer.fit_transform(train_df["text"].astype(str))
X_test=vectorizer.transform(test_df["text"].astype(str))
y_train=train_df["sentiment"]
y_test=test_df["sentiment"]

## Logistic Regression
Mô hình phân loại cơ bản, nhanh và thường được dùng làm baseline.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
lr=LogisticRegression(max_iter=1000)
lr.fit(X_train,y_train)
lr_pred=lr.predict(X_test)
print(classification_report(y_test,lr_pred))

## Multinomial Naive Bayes
Thuật toán cổ điển rất phổ biến cho Text Classification.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
nb=MultinomialNB()
nb.fit(X_train,y_train)
nb_pred=nb.predict(X_test)
print(classification_report(y_test,nb_pred))

## Support Vector Machine (SVM)
Một trong những mô hình mạnh nhất cho dữ liệu văn bản truyền thống.

In [ ]:
from sklearn.svm import LinearSVC
svm=LinearSVC()
svm.fit(X_train,y_train)
svm_pred=svm.predict(X_test)
print(classification_report(y_test,svm_pred))

# Part C - Large Language Model (LLM)
Sử dụng mô hình RoBERTa được huấn luyện sẵn cho Sentiment Analysis trên Twitter.

In [ ]:
from transformers import pipeline
pipe=pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

## Thử nghiệm nhanh
Kiểm tra mô hình đã hoạt động đúng hay chưa.

In [ ]:
pipe("I love this movie")

## Đánh giá trên tập Test
Chạy mô hình trên toàn bộ dữ liệu test và tính các chỉ số đánh giá.

In [ ]:
from tqdm import tqdm
llm_pred=[]
for text in tqdm(test_df["text"].astype(str)):
    result=pipe(text)[0]
    llm_pred.append(result["label"].lower())
print(classification_report(y_test,llm_pred))

# Part D - So sánh mô hình
So sánh Accuracy giữa các thuật toán.

In [ ]:
from sklearn.metrics import accuracy_score
results=pd.DataFrame({"Model":["Logistic Regression","Naive Bayes","SVM"],"Accuracy":[accuracy_score(y_test,lr_pred),accuracy_score(y_test,nb_pred),accuracy_score(y_test,svm_pred)]})
results

In [ ]:
sns.barplot(data=results,x="Model",y="Accuracy")
plt.title("Model Comparison")
plt.show()

## Conclusion
✅ EDA

✅ TF-IDF Feature Extraction

✅ Logistic Regression

✅ Naive Bayes

✅ SVM

✅ RoBERTa LLM

✅ So sánh kết quả các mô hình để lựa chọn phương pháp phù hợp nhất.